In [1]:
import pandas as pd
import json
from sklearn.metrics import classification_report, mean_absolute_error


In [5]:
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score


def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def normalize_gold(gold_path):
    rows = []
    for obj in read_jsonl(gold_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "gold_rating": obj.get("rating"),
                "gold_comment": obj.get("comment"),
            }
        )
    return pd.DataFrame(rows)


def normalize_pred(pred_path):
    rows = []
    for obj in read_jsonl(pred_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "llm_rating": obj.get("rating"),
                "llm_comment": obj.get("comment"),
                "status": obj.get("status", "success"),
            }
        )
    df = pd.DataFrame(rows)
    if "status" in df.columns:
        df = df[df["status"] == "success"].copy()
    return df


def infer_dataset_name(folder_path):
    return Path(folder_path).name


def infer_model_name(pred_file_name):
    name = pred_file_name.replace(".jsonl", "")
    if name.startswith("gemini"):
        return "gemini"
    if name.startswith("qwen"):
        return "qwen"
    if name.startswith("claude"):
        return "claude"
    return name


def evaluate_one_pair(gold_path, pred_path):
    df_gold = normalize_gold(gold_path)
    df_pred = normalize_pred(pred_path)

    df = pd.merge(df_gold, df_pred, on="article_id", how="inner")

    total_merged = len(df)
    df = df.dropna(subset=["gold_rating", "llm_rating"]).copy()
    valid_rows = len(df)

    if valid_rows == 0:
        return None, None

    df["gold_rating"] = df["gold_rating"].astype(int)
    df["llm_rating"] = df["llm_rating"].astype(int)
    df["abs_error"] = (df["gold_rating"] - df["llm_rating"]).abs()

    labels = [-2, -1, 0, 1, 2]

    metrics = {
        "dataset": infer_dataset_name(Path(gold_path).parent),
        "model": infer_model_name(Path(pred_path).name),
        "gold_file": Path(gold_path).name,
        "pred_file": Path(pred_path).name,
        "gold_rows": len(df_gold),
        "pred_rows_success": len(df_pred),
        "merged_rows": total_merged,
        "valid_rows": valid_rows,
        "coverage_vs_gold": round(valid_rows / max(len(df_gold), 1), 4),
        "mae": round(mean_absolute_error(df["gold_rating"], df["llm_rating"]), 4),
        "accuracy": round(accuracy_score(df["gold_rating"], df["llm_rating"]), 4),
        "macro_f1": round(f1_score(df["gold_rating"], df["llm_rating"], labels=labels, average="macro", zero_division=0), 4),
    }

    return metrics, df


def run_batch_evaluation(
    base_dir="..",
    summary_csv="evaluation_summary.csv",
    details_dir="evaluation_details",
):
    base = Path(base_dir)
    out_dir = base / details_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    summary_rows = []

    gold_files = list(base.rglob("gold_standard*.jsonl"))

    for gold_file in gold_files:
        folder = gold_file.parent

        pred_files = [
            p for p in folder.glob("*.jsonl")
            if p.name != gold_file.name and not p.name.startswith("gold_standard")
        ]

        for pred_file in pred_files:
            metrics, merged_df = evaluate_one_pair(gold_file, pred_file)
            if metrics is None:
                continue

            summary_rows.append(metrics)

            details_name = f"{metrics['dataset']}__{metrics['model']}__details.csv"
            merged_df.to_csv(out_dir / details_name, index=False, encoding="utf-8-sig")

    summary_df = pd.DataFrame(summary_rows)

    if len(summary_df) == 0:
        print("No valid evaluation pairs found.")
        return pd.DataFrame()

    summary_df = summary_df.sort_values(["dataset", "macro_f1"], ascending=[True, False]).reset_index(drop=True)
    summary_df.to_csv(base / summary_csv, index=False, encoding="utf-8-sig")

    print("Saved summary:", str(base / summary_csv))
    print("Saved details folder:", str(out_dir))
    print("Total evaluated pairs:", len(summary_df))

    return summary_df


# Quick run from this notebook folder:
summary = run_batch_evaluation(
    base_dir="..",
    summary_csv="evaluation_summary.csv",
    details_dir="evaluation_details",
)

summary.head(20)

Saved summary: ../evaluation_summary.csv
Saved details folder: ../evaluation_details
Total evaluated pairs: 47


,dataset,model,gold_file,pred_file,gold_rows,pred_rows_success,merged_rows,valid_rows,coverage_vs_gold,mae,accuracy,macro_f1
0,apolitical_relgious_hefajot,claude,gold_standard_hefajot.jsonl,claude_sonet_5.jsonl,18,19,18,18,1.0000,0.0000,1.0000,0.2000
1,apolitical_relgious_hefajot,gemini,gold_standard_hefajot.jsonl,gemini3.1_pro_ext_hefajot.jsonl,18,18,18,18,1.0000,0.0000,1.0000,0.2000
2,apolitical_relgious_hefajot,qwen,gold_standard_hefajot.jsonl,qwen2.5_7b_hefajot.jsonl,18,17,17,17,0.9444,1.2941,0.1765,0.0600
3,apolitical_relgious_iskon,gemini,gold_standard_iskon.jsonl,gemini3.1_pro_ext_iskon.jsonl,8,8,8,8,1.0000,0.2500,0.7500,0.1714
4,apolitical_relgious_iskon,claude,gold_standard_iskon.jsonl,claude_sonnet_5_iskon.jsonl,8,8,8,8,1.0000,0.3750,0.6250,0.1538
5,apolitical_relgious_iskon,qwen,gold_standard_iskon.jsonl,qwen2.5_7b_iskon.jsonl,8,7,7,7,0.8750,1.7143,0.0000,0.0000
6,apolitical_religious_tablig,qwen,gold_standard_tablig.jsonl,qwen2.5_7b_tablig.jsonl,9,8,8,8,0.8889,0.2500,0.7500,0.3214
7,apolitical_religious_tablig,gemini,gold_standard_tablig.jsonl,gemini3.1_pro_ext_tablig.jsonl,9,10,9,9,1.0000,0.2222,0.7778,0.3000
8,apolitical_religious_tablig,claude,gold_standard_tablig.jsonl,claude_sonnet_5_tablig.jsonl,9,10,9,9,1.0000,0.5556,0.4444,0.1455
9,cultural_chayanot,gemini,gold_standard_chayanot.jsonl,gemini3.1_pro_ext_chayanot.jsonl,12,12,12,12,1.0000,0.1667,0.8333,0.2800


In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def build_presentation_outputs(
    summary_csv="../evaluation_summary.csv",
    reports_dir="../evaluation_reports",
):
    summary_path = Path(summary_csv)
    reports_path = Path(reports_dir)
    reports_path.mkdir(parents=True, exist_ok=True)

    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file not found: {summary_path}")

    df = pd.read_csv(summary_path)

    # 1) Clean table for slides
    slide_table = df[[
        "dataset", "model", "valid_rows", "coverage_vs_gold", "mae", "accuracy", "macro_f1"
    ]].sort_values(["dataset", "macro_f1"], ascending=[True, False])
    slide_table.to_csv(reports_path / "slide_table.csv", index=False, encoding="utf-8-sig")

    # 2) Dataset x Model pivot for quick comparison
    pivot_f1 = df.pivot_table(index="dataset", columns="model", values="macro_f1", aggfunc="mean")
    pivot_f1.to_csv(reports_path / "macro_f1_pivot.csv", encoding="utf-8-sig")

    # 3) Average score by model
    model_avg = df.groupby("model", as_index=False).agg(
        avg_macro_f1=("macro_f1", "mean"),
        avg_mae=("mae", "mean"),
        avg_accuracy=("accuracy", "mean"),
    ).sort_values("avg_macro_f1", ascending=False)
    model_avg.to_csv(reports_path / "model_average_scores.csv", index=False, encoding="utf-8-sig")

    # 4) Chart: Average Macro F1 by model
    plt.figure(figsize=(10, 5))
    plt.bar(model_avg["model"], model_avg["avg_macro_f1"])
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("Average Macro F1")
    plt.title("Average Macro F1 by Model")
    plt.tight_layout()
    plt.savefig(reports_path / "avg_macro_f1_by_model.png", dpi=200)
    plt.close()

    # 5) Chart: MAE vs Macro F1 scatter
    plt.figure(figsize=(8, 6))
    plt.scatter(df["mae"], df["macro_f1"])
    plt.xlabel("MAE (lower is better)")
    plt.ylabel("Macro F1 (higher is better)")
    plt.title("MAE vs Macro F1 across all dataset-model pairs")
    plt.tight_layout()
    plt.savefig(reports_path / "mae_vs_macro_f1.png", dpi=200)
    plt.close()

    print("Saved report folder:", reports_path)
    print("Files:")
    for p in sorted(reports_path.glob("*")):
        print("-", p.name)

    return slide_table, model_avg, pivot_f1


slide_table, model_avg, pivot_f1 = build_presentation_outputs(
    summary_csv="../evaluation_summary.csv",
    reports_dir="../evaluation_reports",
)

slide_table.head(15)

Matplotlib is building the font cache; this may take a moment.


Saved report folder: ../evaluation_reports
Files:
- avg_macro_f1_by_model.png
- macro_f1_pivot.csv
- mae_vs_macro_f1.png
- model_average_scores.csv
- slide_table.csv


,dataset,model,valid_rows,coverage_vs_gold,mae,accuracy,macro_f1
0,apolitical_relgious_hefajot,claude_sonet_5,18,1.0000,0.0000,1.0000,0.2000
1,apolitical_relgious_hefajot,gemini,18,1.0000,0.0000,1.0000,0.2000
2,apolitical_relgious_hefajot,qwen,17,0.9444,1.2941,0.1765,0.0600
3,apolitical_relgious_iskon,gemini,8,1.0000,0.2500,0.7500,0.1714
4,apolitical_relgious_iskon,claude_sonnet_5_iskon,8,1.0000,0.3750,0.6250,0.1538
5,apolitical_relgious_iskon,qwen,7,0.8750,1.7143,0.0000,0.0000
6,apolitical_religious_tablig,qwen,8,0.8889,0.2500,0.7500,0.3214
7,apolitical_religious_tablig,gemini,9,1.0000,0.2222,0.7778,0.3000
8,apolitical_religious_tablig,claude_sonnet_5_tablig,9,1.0000,0.5556,0.4444,0.1455
9,cultural_chayanot,gemini,12,1.0000,0.1667,0.8333,0.2800


In [6]:
from pathlib import Path
import pandas as pd


def export_category_views(summary_csv="../evaluation_summary.csv", reports_dir="../evaluation_reports"):
    summary_path = Path(summary_csv)
    reports_path = Path(reports_dir)
    reports_path.mkdir(parents=True, exist_ok=True)

    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file not found: {summary_path}")

    df = pd.read_csv(summary_path)

    geo_df = df[df["dataset"].str.startswith("geo_", na=False)].copy()
    plt_df = df[df["dataset"].str.startswith("plt_", na=False)].copy()

    geo_df = geo_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True])
    plt_df = plt_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True])

    geo_df.to_csv(reports_path / "geo_summary.csv", index=False, encoding="utf-8-sig")
    plt_df.to_csv(reports_path / "plt_summary.csv", index=False, encoding="utf-8-sig")

    # Best model per dataset for quick slide/table
    best_geo = geo_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True]).groupby("dataset", as_index=False).first()
    best_plt = plt_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True]).groupby("dataset", as_index=False).first()

    best_geo.to_csv(reports_path / "best_geo_models.csv", index=False, encoding="utf-8-sig")
    best_plt.to_csv(reports_path / "best_plt_models.csv", index=False, encoding="utf-8-sig")

    print("Geo rows:", len(geo_df), "| Geo datasets:", geo_df["dataset"].nunique())
    print("Plt rows:", len(plt_df), "| Plt datasets:", plt_df["dataset"].nunique())
    print("Saved files:")
    print("-", reports_path / "geo_summary.csv")
    print("-", reports_path / "plt_summary.csv")
    print("-", reports_path / "best_geo_models.csv")
    print("-", reports_path / "best_plt_models.csv")

    return geo_df, plt_df, best_geo, best_plt


geo_df, plt_df, best_geo, best_plt = export_category_views(
    summary_csv="../evaluation_summary.csv",
    reports_dir="../evaluation_reports",
)

best_geo, best_plt

Geo rows: 14 | Geo datasets: 4
Plt rows: 8 | Plt datasets: 4
Saved files:
- ../evaluation_reports/geo_summary.csv
- ../evaluation_reports/plt_summary.csv
- ../evaluation_reports/best_geo_models.csv
- ../evaluation_reports/best_plt_models.csv


(                dataset   model                      gold_file  \
 0             geo_india  gemini      gold_standard_india.jsonl   
 1  geo_israel_palestine  gemini  gold_standard_palestine.jsonl   
 2          geo_pakistan  gemini   gold_standard_pakistan.jsonl   
 3              geo_west  gemini       gold_standard_west.jsonl   
 
                                pred_file  gold_rows  pred_rows_success  \
 0     gemini3.1_pro_extended_india.jsonl         50                 50   
 1      gemini3.1_pro_ext_palestine.jsonl         10                 10   
 2  gemini3.1_pro_extended_pakistan.jsonl         41                 43   
 3           gemini3.1_pro_ext_west.jsonl         24                 24   
 
    merged_rows  valid_rows  coverage_vs_gold     mae  accuracy  macro_f1  
 0           50          50               1.0  0.0400    0.9600    0.7613  
 1           10          10               1.0  0.0000    1.0000    0.6000  
 2           41          41               1.0  0.0732    0